In [10]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType

In [11]:
MODEL_NAME = "/workspace/qwen3-4b"
TRAIN_DATA_PATH = "./valid.jsonl"   # 训练集文件
EVAL_DATA_PATH = "./test.jsonl"     # 独立验证集文件
OUTPUT_DIR = "./sea_lion_8b_medical_lora"
MAX_SEQ_LEN = 1024

# LoRA超参
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True  # 强制只读取本地模型，不联网
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 8bit量化配置
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,                # 开启8bit量化
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
    llm_int8_skip_modules=None,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    # quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [13]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    inference_mode=False
)

# 强制物化所有 meta 参数
#dummy_input = torch.ones(1, 1, dtype=torch.long, device="cuda")
#_ = model(dummy_input)   # 这步会触发参数加载

model = get_peft_model(model, lora_config)

# 再移一次到 GPU（确保）
# model = model.to("cuda") 

model.print_trainable_parameters()

trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


In [14]:
def format_prompt(sample):
    # 去除多余缩进空格，避免无效token
    template_prefix = """
        ### Instruction:{instr}
        ### Input:
        {inp}
        ### Response:
    """
    template_full = template_prefix + "{resp}" + tokenizer.eos_token

    full_text = template_full.format(
        instr=sample["instruction"],
        inp=sample["input"],
        resp=sample["output"]
    )
    # 单独编码前缀，获取长度用于mask loss
    prefix_text = template_prefix.format(
        instr=sample["instruction"],
        inp=sample["input"]
    )
    prefix_tokens = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    return {"text": full_text, "prefix_len": len(prefix_tokens)}

def tokenize_fn(sample):
    res = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length"
    )
    res["prefix_len"] = sample["prefix_len"]
    return res

def set_labels(sample):
    # 转torch tensor
    input_ids = torch.tensor(sample["input_ids"])
    labels = input_ids.clone()
    
    prefix_len = sample["prefix_len"]
    # 屏蔽前缀
    labels[:prefix_len] = -100
    # 屏蔽padding
    mask = torch.tensor(sample["attention_mask"])
    labels[mask == 0] = -100
    
    # 转回list存入dataset（datasets库只能存原生类型）
    sample["labels"] = labels.tolist()
    return sample

def process_dataset(file_path):
    # 加载单份jsonl数据集
    ds = load_dataset("json", data_files=file_path, split="train")
    # 格式化文本+计算前缀长度
    ds = ds.map(format_prompt)
    # 分词
    ds = ds.map(tokenize_fn, remove_columns=ds.column_names)
    # 生成掩码labels
    ds = ds.map(set_labels)
    # 转为torch张量格式
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


In [15]:
# 分别处理训练集、独立验证集
tokenized_train_ds = process_dataset(TRAIN_DATA_PATH)
tokenized_eval_ds = process_dataset(EVAL_DATA_PATH)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    optim="adamw_torch",  # 原生优化器，不依赖bitsandbytes
    report_to="none",
    save_total_limit=5,

    # 评估相关
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)


In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_eval_ds,  # 传入独立验证集
)

In [18]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.407166,2.660058
100,2.721817,2.605379
150,2.241555,2.594615
200,2.552398,2.560840
250,2.249939,2.556450
300,1.920401,2.551932
350,2.466929,2.540402
400,2.327366,2.531275
450,2.015306,2.539015
500,1.768480,2.527324


TrainOutput(global_step=3000, training_loss=2.081018691062927, metrics={'train_runtime': 3758.3085, 'train_samples_per_second': 0.798, 'train_steps_per_second': 0.798, 'total_flos': 6.7081608364032e+16, 'train_loss': 2.081018691062927, 'epoch': 3.0})

In [19]:
trainer.save_model(f"{OUTPUT_DIR}/final_lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_lora")

('./sea_lion_8b_medical_lora/final_lora/tokenizer_config.json',
 './sea_lion_8b_medical_lora/final_lora/chat_template.jinja',
 './sea_lion_8b_medical_lora/final_lora/tokenizer.json')